# Geospatial POI and Routing Features

## Objective

This notebook develops external geographic features for Victorian
rental properties, including:

- distance to nearest train station
- distance to Melbourne CBD
- proximity to schools
- proximity to parks
- proximity to shopping/amenities

Straight-line distance is used as an initial baseline.
OpenRouteService route distance will subsequently be calculated
at suburb level to reduce API usage.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import time
import requests

from pathlib import Path
from sklearn.neighbors import BallTree

## 1. File paths

Define the locations of the raw and curated datasets.

In [ ]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

TRANSPORT_DIR = RAW_DIR / "transport"
SCHOOL_DIR = RAW_DIR / "schools"
OSM_DIR = RAW_DIR / "osm"

CURATED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Victorian train stations

Transport Victoria GTFS data are divided by transport mode.

The metropolitan train dataset and regional train dataset are combined
so that train accessibility can be calculated for properties throughout Victoria.

Only station-level locations are retained where possible, rather than
individual platforms or entrances.

In [ ]:
metro_train_path = (
    TRANSPORT_DIR
    / "gtfs"
    / "1"
    / "google_transit"
    / "stops.txt"
)

regional_train_path = (
    TRANSPORT_DIR
    / "gtfs"
    / "2"
    / "google_transit"
    / "stops.txt"
)

metro_stops = pd.read_csv(metro_train_path)
regional_stops = pd.read_csv(regional_train_path)

print("Metro stop records:", len(metro_stops))
print("Regional stop records:", len(regional_stops))

In [ ]:
train_stops = pd.concat(
    [metro_stops, regional_stops],
    ignore_index=True
)

train_stops.head()

In [ ]:
if "location_type" in train_stops.columns:
    stations = train_stops[
        train_stops["location_type"] == 1
    ].copy()
else:
    stations = train_stops.copy()


stations = stations[
    [
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ]
].copy()

stations = stations.rename(columns={
    "stop_lat": "latitude",
    "stop_lon": "longitude"
})


stations = stations.dropna(
    subset=["latitude", "longitude"]
)

stations = stations.drop_duplicates(
    subset=[
        "stop_name",
        "latitude",
        "longitude"
    ]
).reset_index(drop=True)

print("Number of train stations:", len(stations))

stations.head(10)


## 3. Victorian school locations

The Victorian School Locations dataset contains primary and secondary
school locations throughout Victoria.

School coordinates are used to calculate:

- distance to the nearest school
- number of schools within 2 km

These variables are used as measures of access to education and local
liveability.

In [ ]:
school_path = (
    SCHOOL_DIR
    / "dv402-SchoolLocations2025.csv"
)

schools = pd.read_csv(school_path)

print("Number of school records:", len(schools))

schools.head()

In [ ]:
schools = schools[
    [
        "School_No",
        "School_Name",
        "School_Type",
        "Education_Sector",
        "Address_Town",
        "Address_Postcode",
        "X",
        "Y"
    ]
].copy()

schools = schools.rename(columns={
    "X": "longitude",
    "Y": "latitude"
})

schools = schools.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)


## 4. Temporary locations for pipeline development

A small set of Victorian suburbs is used to test the geographic feature
pipeline before the group's complete rental property dataset is available.

These coordinates are temporary development values and will later be
replaced with property coordinates or suburb centroids.

In [ ]:
# Load 2025 rental listings
rentals_2025_path = RAW_DIR / "domain" / "Data" / "vic_rentals_all.csv"

rentals_2025 = pd.read_csv(rentals_2025_path)

print("Listings:", len(rentals_2025))
print("Suburbs:", rentals_2025["suburb"].nunique())

rentals_2025.head()

In [ ]:
# Create one representative location per suburb
suburb_locations = (
    rentals_2025
    .dropna(subset=["suburb", "lat", "lon"])
    .groupby("suburb", as_index=False)
    .agg(
        latitude=("lat", "median"),
        longitude=("lon", "median"),
        listing_count=("listing_id", "count"),
        median_weekly_rent=("weekly_rent", "median")
    )
)

# Keep the existing notebook code working without changing every cell
test_locations = suburb_locations.copy()

print("Suburbs:", len(test_locations))

test_locations.head()

## 5. Straight-line distance to points of interest

Nearest points of interest are identified using a BallTree with the
Haversine distance metric.

This provides an efficient measure of straight-line geographic distance
between suburb/property locations and external amenities.

In [ ]:
EARTH_RADIUS_KM = 6371.0088

def add_nearest_poi(
    locations,
    pois,
    poi_name=None,
    prefix="poi",
    keep_coordinates=False
):

    locations = locations.copy()
    pois = pois.copy()

    location_coords = np.radians(
        locations[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    poi_coords = np.radians(
        pois[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    tree = BallTree(
        poi_coords,
        metric="haversine"
    )

    distances, indices = tree.query(
        location_coords,
        k=1
    )

    nearest_indices = indices[:, 0]

    locations[f"{prefix}_distance_km"] = (
        distances[:, 0]
        * EARTH_RADIUS_KM
    )

    if poi_name is not None:
        locations[f"nearest_{prefix}"] = (
            pois.iloc[nearest_indices][poi_name]
            .to_numpy()
        )

    if keep_coordinates:
        locations[f"{prefix}_latitude"] = (
            pois.iloc[nearest_indices]["latitude"]
            .to_numpy()
        )

        locations[f"{prefix}_longitude"] = (
            pois.iloc[nearest_indices]["longitude"]
            .to_numpy()
        )

    return locations

## 6. Distance to nearest train station

For each location, the closest metropolitan or regional train station
is identified.

Straight-line distance is used initially as the baseline accessibility
measure.

In [ ]:
test_locations = add_nearest_poi(
    locations=test_locations,
    pois=stations,
    poi_name="stop_name",
    prefix="train",
    keep_coordinates=True
)

test_locations[
    [
        "suburb",
        "nearest_train",
        "train_distance_km",
        "train_latitude",
        "train_longitude"
    ]
]

## 7. Distance to nearest school

The closest Victorian school is identified for each location.

This feature measures access to nearby educational facilities.

In [ ]:
test_locations = add_nearest_poi(
    locations=test_locations,
    pois=schools,
    poi_name="School_Name",
    prefix="school"
)

test_locations[
    [
        "suburb",
        "nearest_school",
        "school_distance_km"
    ]
]

## 8. Number of nearby points of interest

In addition to nearest-distance measures, amenities within a specified
radius are counted.

For schools, a 2 km radius is used initially as a measure of local
educational accessibility.

In [ ]:
def count_pois_within_radius(
    locations,
    pois,
    radius_km,
    prefix
):

    locations = locations.copy()

    location_coords = np.radians(
        locations[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    poi_coords = np.radians(
        pois[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    tree = BallTree(
        poi_coords,
        metric="haversine"
    )

    radius_radians = (
        radius_km
        / EARTH_RADIUS_KM
    )

    neighbours = tree.query_radius(
        location_coords,
        r=radius_radians
    )

    locations[
        f"{prefix}_within_{radius_km:g}km"
    ] = [
        len(points)
        for points in neighbours
    ]

    return locations

In [ ]:
test_locations = count_pois_within_radius(
    locations=test_locations,
    pois=schools,
    radius_km=2,
    prefix="schools"
)

test_locations[
    [
        "suburb",
        "nearest_school",
        "school_distance_km",
        "schools_within_2km"
    ]
]

## 9. Distance to Melbourne CBD

Distance to Melbourne CBD is used as a measure of accessibility to the
central employment and commercial area.

Straight-line distance is calculated first. Route-based distance will
later be calculated using OpenRouteService.

In [ ]:
CBD = pd.DataFrame({
    "name": [
        "Melbourne CBD"
    ],

    "latitude": [
        -37.8136
    ],

    "longitude": [
        144.9631
    ]
})

test_locations = add_nearest_poi(
    locations=test_locations,
    pois=CBD,
    poi_name="name",
    prefix="cbd"
)

test_locations[
    [
        "suburb",
        "cbd_distance_km"
    ]
]

## 10. Initial geographic feature table

The following table combines the initial geographic accessibility
features created from the external datasets.

In [ ]:
test_locations[
    [
        "suburb",

        "nearest_train",
        "train_distance_km",

        "nearest_school",
        "school_distance_km",
        "schools_within_2km",

        "cbd_distance_km"
    ]
]

## 11. OpenRouteService setup

Straight-line distance does not account for the actual road network.
OpenRouteService (ORS) is therefore used to calculate route-based
distances to the nearest train station and Melbourne CBD.

Routing is performed at suburb level rather than for every individual
rental listing to reduce the number of API requests.

In [ ]:
import os
import requests

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / "ors_config.env")

ORS_API_KEY = os.getenv("ORS_API_KEY")

if ORS_API_KEY is None:
    raise ValueError("ORS_API_KEY was not found.")

print("ORS API key loaded successfully.")

## 12. Route distance function

A reusable function is created to calculate driving distance between
two geographic coordinates using OpenRouteService.

OpenRouteService requires coordinates in longitude-latitude order.

In [ ]:
ORS_URL = (
    "https://api.heigit.org/openrouteservice/v2/"
    "directions/driving-car"
)


def get_route_distance_km(
    origin_lon,
    origin_lat,
    destination_lon,
    destination_lat
):
    headers = {
        "Authorization": ORS_API_KEY,
        "Content-Type": "application/json"
    }

    body = {
        "coordinates": [
            [origin_lon, origin_lat],
            [destination_lon, destination_lat]
        ]
    }

    response = requests.post(
        ORS_URL,
        json=body,
        headers=headers
    )

    response.raise_for_status()

    route = response.json()

    distance_metres = (
        route["routes"][0]
        ["summary"]
        ["distance"]
    )

    return distance_metres / 1000

## 13. Test route calculation

A single route is calculated first to verify that the
OpenRouteService API connection is functioning correctly.

In [ ]:
test_route = get_route_distance_km(
    origin_lon=144.967,
    origin_lat=-37.800,
    destination_lon=144.9631,
    destination_lat=-37.8136
)

print(
    f"Test route distance: "
    f"{test_route:.2f} km"
)

## 14. Driving distance to Melbourne CBD

Driving distance to Melbourne CBD is calculated in addition to the
straight-line distance.

This provides a more realistic accessibility measure because it
accounts for the available road network.

In [357]:
CBD_LAT = -37.8136
CBD_LON = 144.9631

ORS_MATRIX_URL = (
    "https://api.heigit.org/openrouteservice/v2/matrix/driving-car"
)

def get_cbd_route_distances(locations, batch_size=50):
    results = []

    headers = {
        "Authorization": ORS_API_KEY,
        "Content-Type": "application/json"
    }

    for start in range(0, len(locations), batch_size):
        batch = locations.iloc[start:start + batch_size]

        # First location = CBD
        coordinates = [[CBD_LON, CBD_LAT]]

        # Remaining locations = suburb coordinates
        coordinates += [
            [row["longitude"], row["latitude"]]
            for _, row in batch.iterrows()
        ]

        body = {
            "locations": coordinates,

            # Suburbs are sources
            "sources": [str(i) for i in range(1, len(coordinates))],

            # CBD is destination
            "destinations": ["0"],

            "metrics": ["distance"],
            "units": "km"
        }

        response = requests.post(
            ORS_MATRIX_URL,
            json=body,
            headers=headers
        )

        response.raise_for_status()

        data = response.json()

        # Each source has one destination (CBD)
        results.extend(
            row[0] if row else np.nan
            for row in data["distances"]
        )

    return results

test_locations["cbd_route_km"] = (
    get_cbd_route_distances(test_locations)
)

test_locations[
    [
        "suburb",
        "cbd_distance_km",
        "cbd_route_km"
    ]
].head(20)

,suburb,cbd_distance_km,cbd_route_km
0,ABBOTSFORD,3.390267,3.89
1,ABERFELDIE,8.382840,9.43
2,AIRPORT WEST,12.288182,15.03
3,ALBANVALE,18.607729,28.15
4,ALBERT PARK,3.277531,4.88
5,ALBION,12.720562,18.91
6,ALEXANDRA,95.563036,162.38
7,ALFREDTON,106.253668,125.48
8,ALLANSFORD,216.336721,245.37
9,ALPHINGTON,6.432093,8.92


## 15. Driving distance to nearest train station

The nearest train station is first identified using straight-line
distance. OpenRouteService is then used to calculate road distance
between each suburb location and its identified nearest station.

This two-stage approach avoids routing to every train station.

In [358]:
def get_train_route_distances(locations, batch_size=10):
    results = []

    headers = {
        "Authorization": ORS_API_KEY,
        "Content-Type": "application/json"
    }

    for start in range(0, len(locations), batch_size):

        batch = locations.iloc[start:start + batch_size]

        origins = [
            [row["longitude"], row["latitude"]]
            for _, row in batch.iterrows()
        ]

        destinations = [
            [row["train_longitude"], row["train_latitude"]]
            for _, row in batch.iterrows()
        ]

        coordinates = origins + destinations
        n = len(batch)

        body = {
            "locations": coordinates,
            "sources": [str(i) for i in range(n)],
            "destinations": [str(i) for i in range(n, 2 * n)],
            "metrics": ["distance"],
            "units": "km"
        }

        # Keep retrying if ORS temporarily rate-limits us
        while True:

            response = requests.post(
                ORS_MATRIX_URL,
                json=body,
                headers=headers
            )

            if response.status_code == 200:
                break

            elif response.status_code == 429:
                print("Rate limit reached. Waiting 60 seconds...")
                time.sleep(60)

            else:
                print(
                    f"Failed batch {start}-{start + n}:",
                    response.status_code,
                    response.text
                )
                response.raise_for_status()

        matrix = response.json()["distances"]

        results.extend(
            matrix[i][i]
            for i in range(n)
        )

        # Small delay between successful requests
        time.sleep(2)

    return results


test_locations["train_route_km"] = (
    get_train_route_distances(test_locations, batch_size=10)
)

In [359]:
test_locations[
    [
        "suburb",
        "nearest_train",
        "train_distance_km",
        "train_route_km"
    ]
].head(20)

,suburb,nearest_train,train_distance_km,train_route_km
0,ABBOTSFORD,North Richmond Railway Station,0.784608,0.98
1,ABERFELDIE,Essendon Railway Station,1.274584,1.40
2,AIRPORT WEST,Oak Park Railway Station,3.587241,8.86
3,ALBANVALE,St Albans Railway Station (St Albans),2.633462,3.08
4,ALBERT PARK,Anzac Railway Station,1.917232,2.50
5,ALBION,Albion Railway Station,0.521667,0.89
6,ALEXANDRA,Euroa Railway Station,50.763521,64.43
7,ALFREDTON,Wendouree Railway Station,2.824327,5.02
8,ALLANSFORD,Sherwood Park Railway Station,5.364911,6.63
9,ALPHINGTON,Alphington Railway Station,0.771176,2.37


## 16. Geographic visualisation

A map is created to visualise the temporary suburb locations and
Victorian train stations used in the accessibility analysis.

In [360]:

from IPython.display import display

import folium

m = folium.Map(
    location=[-37.81, 144.96],
    zoom_start=9
)

for _, row in test_locations.iterrows():

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=5,
        tooltip=row["suburb"],
        fill=True
    ).add_to(m)

for _, row in stations.iterrows():

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=2,
        tooltip=row["stop_name"],
        fill=True
    ).add_to(m)

display(m)

## 17. Parks and open-space accessibility

The OpenStreetMap Victoria dataset will be used to extract park and
open-space locations. These will be used to construct proximity and
local amenity-count features.

In [361]:
from pyrosm import OSM

osm_path = OSM_DIR / "victoria-260908.osm.pbf"

print("OSM path:", osm_path)
print("Exists:", osm_path.exists())

osm = OSM(str(osm_path))

OSM path: ../data/raw/osm/victoria-260908.osm.pbf
Exists: True


In [362]:
park_filter = {
    "leisure": ["park", "garden", "nature_reserve"]
}

parks_gdf = osm.get_pois(
    custom_filter=park_filter
)

print("Raw park features:", len(parks_gdf))

parks_gdf.head()

# Remove features without geometry
parks_gdf = parks_gdf[
    parks_gdf.geometry.notna()
].copy()

# Project to a metric CRS before calculating centroids
parks_projected = parks_gdf.to_crs(epsg=7855)

# Represent each park by its centroid
parks_projected["geometry"] = (
    parks_projected.geometry.centroid
)

# Convert back to latitude/longitude
parks_points = parks_projected.to_crs(epsg=4326)

parks = pd.DataFrame({
    "park_name": parks_points["name"],
    "latitude": parks_points.geometry.y,
    "longitude": parks_points.geometry.x
})

parks["park_name"] = parks["park_name"].fillna(
    "Unnamed park"
)

parks = parks.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)

print("Usable park records:", len(parks))

parks.head()

Raw park features: 25553
Usable park records: 25553


,park_name,latitude,longitude
0,Unnamed park,-37.832640,145.021231
1,Katandra Football Netball Club,-36.228082,145.560527
2,Websters Reserve,-37.747787,145.155588
3,Number Two Creek Reserve,-37.523376,145.355380
4,Azalea Garden,-37.547945,143.821110


In [363]:
print("Park records before removing duplicates:", len(parks))

print(
    "Unique park coordinates:",
    parks[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Unique named parks:",
    parks.loc[
        parks["park_name"] != "Unnamed park",
        "park_name"
    ].nunique()
)

parks = (
    parks
    .drop_duplicates(
        subset=["latitude", "longitude"]
    )
    .reset_index(drop=True)
)

print("Park records after removing duplicates:", len(parks))

Park records before removing duplicates: 25553
Unique park coordinates: 25551
Unique named parks: 10630
Park records after removing duplicates: 25551


In [364]:
test_locations = add_nearest_poi(
    test_locations,
    parks,
    poi_name="park_name",
    prefix="park"
)

test_locations[
    ["suburb", "nearest_park", "park_distance_km"]
]

,suburb,nearest_park,park_distance_km
0,ABBOTSFORD,Unnamed park,0.257281
1,ABERFELDIE,Unnamed park,0.582967
2,AIRPORT WEST,Unnamed park,0.289644
3,ALBANVALE,Unnamed park,0.217114
4,ALBERT PARK,Broadway Tree Reserve,0.181587
...,...,...,...
648,YARRAVILLE,M Zacour Park,0.059592
649,YARRAWONGA,Hammon Park,0.353595
650,YARROWEYAH,Unnamed park,4.935069
651,YEA,Unnamed park,0.068587


In [365]:
test_locations = count_pois_within_radius(
    test_locations,
    parks,
    radius_km=2,
    prefix="parks"
)

test_locations[
    [
        "suburb",
        "nearest_park",
        "park_distance_km",
        "parks_within_2km"
    ]
]

,suburb,nearest_park,park_distance_km,parks_within_2km
0,ABBOTSFORD,Unnamed park,0.257281,164
1,ABERFELDIE,Unnamed park,0.582967,58
2,AIRPORT WEST,Unnamed park,0.289644,68
3,ALBANVALE,Unnamed park,0.217114,78
4,ALBERT PARK,Broadway Tree Reserve,0.181587,140
...,...,...,...,...
648,YARRAVILLE,M Zacour Park,0.059592,99
649,YARRAWONGA,Hammon Park,0.353595,16
650,YARROWEYAH,Unnamed park,4.935069,0
651,YEA,Unnamed park,0.068587,17


## 18. Shopping and amenity accessibility

OpenStreetMap points of interest will be used to identify shopping
and retail amenities. Distance and nearby-amenity counts will be
constructed as additional liveability features.

In [366]:
#limit shops to malls, department stores, and supermarkets only for now
shopping_filter = {
    "shop": ["mall", "department_store", "supermarket"]
}

shopping_gdf = osm.get_pois(
    custom_filter=shopping_filter
)

print("Shopping features:", len(shopping_gdf))
shopping_gdf.head()

Shopping features: 2085


,lon,changeset,id,timestamp,version,lat,visible,tags,addr:country,addr:housenumber,...,operator,phone,ref,website,organic,second_hand,shop,geometry,osm_type,url
0,144.733861,0.0,32193447,1361574238,3,-37.277362,False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,supermarket,POINT (144.73386 -37.27736),node,NaN
1,145.171842,0.0,218028470,1774608595,15,-37.791434,False,"{""brand"":""Coles"",""brand:wikidata"":""Q1108172"",""...",NaN,55,...,Coles Group,NaN,NaN,https://www.coles.com.au/,NaN,NaN,supermarket,POINT (145.17184 -37.79143),node,NaN
2,145.240404,0.0,260559651,1786865274,14,-37.868257,False,"{""brand"":""Coles"",""brand:wikidata"":""Q1108172"",""...",NaN,NaN,...,Coles Group,NaN,NaN,https://www.coles.com.au/,NaN,NaN,supermarket,POINT (145.2404 -37.86826),node,NaN
3,145.164618,0.0,260710476,1693297610,12,-37.834987,False,"{""addr:state"":""VIC"",""addr:suburb"":""Forest Hill...",NaN,270,...,NaN,+61 3 8878 1300,NaN,https://www.target.com.au/store/vic/forest-hil...,NaN,NaN,department_store,POINT (145.16462 -37.83499),node,NaN
4,145.164335,0.0,260710477,1774608595,8,-37.834960,False,"{""brand"":""Coles"",""brand:wikidata"":""Q1108172"",""...",NaN,NaN,...,Coles Group,NaN,NaN,https://www.coles.com.au/,NaN,NaN,supermarket,POINT (145.16433 -37.83496),node,NaN


In [367]:
shopping_gdf = shopping_gdf[
    shopping_gdf.geometry.notna()
].copy()

shopping_projected = shopping_gdf.to_crs(epsg=7855)

shopping_projected["geometry"] = (
    shopping_projected.geometry.centroid
)

shopping_points = shopping_projected.to_crs(epsg=4326)

shopping = pd.DataFrame({
    "shopping_name": shopping_points["name"],
    "latitude": shopping_points.geometry.y,
    "longitude": shopping_points.geometry.x
})

shopping["shopping_name"] = (
    shopping["shopping_name"]
    .fillna("Unnamed shop")
)

shopping = shopping.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)

print("Usable shopping locations:", len(shopping))

shopping.head()

Usable shopping locations: 2085


,shopping_name,latitude,longitude
0,IGA,-37.277362,144.733861
1,Coles,-37.791434,145.171842
2,Coles,-37.868258,145.240404
3,Target,-37.834987,145.164618
4,Coles,-37.834960,145.164335


In [368]:
test_locations = add_nearest_poi(
    test_locations,
    shopping,
    poi_name="shopping_name",
    prefix="shopping"
)

test_locations = count_pois_within_radius(
    test_locations,
    shopping,
    radius_km=2,
    prefix="shopping"
)

test_locations[
    [
        "suburb",
        "nearest_shopping",
        "shopping_distance_km",
        "shopping_within_2km"
    ]
]

,suburb,nearest_shopping,shopping_distance_km,shopping_within_2km
0,ABBOTSFORD,Tân Hung Asian Grocery,0.321801,30
1,ABERFELDIE,Boundy's Supa IGA,0.413135,3
2,AIRPORT WEST,IGA Xpress,0.159846,14
3,ALBANVALE,Aldi,0.694432,5
4,ALBERT PARK,Foodworks,0.077469,19
...,...,...,...,...
648,YARRAVILLE,Coles,0.105268,8
649,YARRAWONGA,K Hub,0.965554,3
650,YARROWEYAH,Cobram village,5.158823,0
651,YEA,Foodworks,0.419518,1


## 19. Initial geospatial feature table

The constructed features combine public transport, education and
central-city accessibility measures.

The temporary suburb locations will later be replaced by locations
derived from the group's complete rental-property dataset.

In [369]:
geo_features = test_locations[
    [
        "suburb",

        "nearest_train",
        "train_distance_km",
        "train_route_km",

        "nearest_school",
        "school_distance_km",
        "schools_within_2km",

        "nearest_park",
        "park_distance_km",
        "parks_within_2km",

        "nearest_shopping",
        "shopping_distance_km",
        "shopping_within_2km",

        "cbd_distance_km",
        "cbd_route_km"
    ]
].copy()

geo_features

,suburb,nearest_train,train_distance_km,train_route_km,nearest_school,school_distance_km,schools_within_2km,nearest_park,park_distance_km,parks_within_2km,nearest_shopping,shopping_distance_km,shopping_within_2km,cbd_distance_km,cbd_route_km
0,ABBOTSFORD,North Richmond Railway Station,0.784608,0.98,Abbotsford Primary School,0.213159,14,Unnamed park,0.257281,164,Tân Hung Asian Grocery,0.321801,30,3.390267,3.89
1,ABERFELDIE,Essendon Railway Station,1.274584,1.40,Ave Maria College,0.337878,16,Unnamed park,0.582967,58,Boundy's Supa IGA,0.413135,3,8.382840,9.43
2,AIRPORT WEST,Oak Park Railway Station,3.587241,8.86,St Christopher's School,0.161134,3,Unnamed park,0.289644,68,IGA Xpress,0.159846,14,12.288182,15.03
3,ALBANVALE,St Albans Railway Station (St Albans),2.633462,3.08,Albanvale Primary School,0.389291,8,Unnamed park,0.217114,78,Aldi,0.694432,5,18.607729,28.15
4,ALBERT PARK,Anzac Railway Station,1.917232,2.50,Albert Park Primary School,0.121117,12,Broadway Tree Reserve,0.181587,140,Foodworks,0.077469,19,3.277531,4.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
648,YARRAVILLE,Yarraville Railway Station,0.686847,1.21,Yarraville Special Developmental School,0.206180,10,M Zacour Park,0.059592,99,Coles,0.105268,8,7.117684,10.12
649,YARRAWONGA,Springhurst Railway Station,45.098666,62.68,Sacred Heart Primary School,0.717147,3,Hammon Park,0.353595,16,K Hub,0.965554,3,220.849218,274.49
650,YARROWEYAH,Shepparton Railway Station,51.915396,65.77,Cobram Primary School,5.733256,0,Unnamed park,4.935069,0,Cobram village,5.158823,0,215.551887,248.32
651,YEA,Tallarook Railway Station,31.292983,38.27,Sacred Heart School,0.310821,3,Unnamed park,0.068587,17,Foodworks,0.419518,1,78.161073,128.99


In [370]:
geo_features.info()
geo_features.isna().sum()
geo_features.describe()

<class 'pandas.DataFrame'>
RangeIndex: 653 entries, 0 to 652
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   suburb                653 non-null    str    
 1   nearest_train         653 non-null    str    
 2   train_distance_km     653 non-null    float64
 3   train_route_km        653 non-null    float64
 4   nearest_school        653 non-null    str    
 5   school_distance_km    653 non-null    float64
 6   schools_within_2km    653 non-null    int64  
 7   nearest_park          653 non-null    str    
 8   park_distance_km      653 non-null    float64
 9   parks_within_2km      653 non-null    int64  
 10  nearest_shopping      653 non-null    str    
 11  shopping_distance_km  653 non-null    float64
 12  shopping_within_2km   653 non-null    int64  
 13  cbd_distance_km       653 non-null    float64
 14  cbd_route_km          653 non-null    float64
dtypes: float64(7), int64(3), str(5)
me

,train_distance_km,train_route_km,school_distance_km,schools_within_2km,park_distance_km,parks_within_2km,shopping_distance_km,shopping_within_2km,cbd_distance_km,cbd_route_km
count,653.000000,653.000000,653.000000,653.000000,653.000000,653.000000,653.000000,653.000000,653.000000,653.000000
mean,13.690166,19.866202,1.546492,5.200613,0.615710,51.810107,2.665474,6.229709,79.480143,98.825191
std,28.373765,39.017459,3.329349,4.618036,1.143147,70.750972,5.468279,9.165113,87.463251,105.615991
min,0.092642,0.090000,0.028952,0.000000,0.019315,0.000000,0.014838,0.000000,0.414901,0.720000
25%,0.870410,1.500000,0.328047,1.000000,0.160017,6.000000,0.405743,1.000000,16.833625,21.890000
50%,2.608332,3.850000,0.577853,4.000000,0.270986,33.000000,0.798632,3.000000,44.205020,57.800000
75%,12.605865,17.070000,0.977649,8.000000,0.460075,70.000000,1.787661,9.000000,111.242801,136.440000
max,189.640270,245.610000,28.458297,20.000000,11.007131,594.000000,55.922678,83.000000,482.133239,552.950000


In [371]:
output_path = CURATED_DIR / "suburb_geospatial_features.csv"

geo_features.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(geo_features))

Saved: ../data/curated/suburb_geospatial_features.csv
Rows: 653


## 20. Assumptions and limitations

- The current analysis uses temporary suburb coordinates for pipeline
  development. These will be replaced with locations derived from the
  complete rental-property dataset.

- Straight-line distance provides a computationally efficient baseline
  but does not represent actual travel distance.

- OpenRouteService driving distance accounts for the road network but
  does not necessarily represent public-transport travel time.

- Route calculations will be performed at suburb level rather than for
  every individual property to reduce API usage. This means variation
  between properties within the same suburb may not be captured.

- School accessibility is represented using nearest-school distance and
  the number of schools within 2 km. The 2 km radius is a modelling
  assumption and may not represent the appropriate accessibility
  threshold for every household.

- Amenity measures depend on the completeness and accuracy of the
  external geographic datasets.